In [1]:
using Pkg
Pkg.activate(".")


  Activating project at `~/Projects/TextEmbeddingExperiments`


In [2]:
using SimilaritySearch, HDF5, JLD2, Glob, Printf, JSON3, LinearAlgebra

In [3]:
using Cobweb

In [28]:
css"""
.left-cell {
  text-align: right;
  vertical-align: text-top;
}
.right-cell {
  text-align: left;
  vertical-align: text-top;
}  
"""

Cobweb.CSS(".left-cell {\n  text-align: right;\n  vertical-align: text-top;\n}\n.right-cell {\n  text-align: left;\n  vertical-align: text-top;\n}  \n")

In [29]:
include("ollama-client.jl")

tokenize (generic function with 1 method)

In [30]:
repolist = glob("Repositorios-Institucionales/data-repos-meta/*.json")

85-element Vector{String}:
 "Repositorios-Institucionales/data-repos-meta/buap.json"
 "Repositorios-Institucionales/data-repos-meta/ccg.json"
 "Repositorios-Institucionales/data-repos-meta/centrogeo.json"
 "Repositorios-Institucionales/data-repos-meta/ciad.json"
 "Repositorios-Institucionales/data-repos-meta/ciatec.json"
 "Repositorios-Institucionales/data-repos-meta/ciateq.json"
 "Repositorios-Institucionales/data-repos-meta/cicese.json"
 "Repositorios-Institucionales/data-repos-meta/cicy.json"
 "Repositorios-Institucionales/data-repos-meta/cide.json"
 "Repositorios-Institucionales/data-repos-meta/cidesi.json"
 ⋮
 "Repositorios-Institucionales/data-repos-meta/unam-ssn-literatura.json"
 "Repositorios-Institucionales/data-repos-meta/unam-ssn.json"
 "Repositorios-Institucionales/data-repos-meta/unam-tic.json"
 "Repositorios-Institucionales/data-repos-meta/unamic-micisan.json"
 "Repositorios-Institucionales/data-repos-meta/unicach.json"
 "Repositorios-Institucionales/data-repos-meta/uqroo

In [ ]:
struct MetaVectorDatabase{MetaType,DatabaseType} <: AbstractDatabase
    meta::MetaType
    key2id::Dict{String,Int}
    emb::DatabaseType
end

function MetaVectorDatabase(
        metafile::AbstractString; 
        embfile = replace(metafile, ".json" => "") * ".h5",
        embkey = "description/emb"
        )
    
    meta = [JSON3.read(line) for line in eachline(metafile)]
    key2id = Dict(r["key"] => i for (i, r) in enumerate(meta))
    emb = MatrixDatabase(h5read(embfile, embkey))
    @assert length(meta) == length(emb)
    MetaVectorDatabase(meta, key2id, emb)
end

function 
end


MetaVectorDatabase

In [81]:
record(db::MetaVectorDatabase, key::String) = db.meta[db.key2id[key]]
record(db::MetaVectorDatabase, i::Integer) = db.meta[i]
Base.getindex(db::MetaVectorDatabase, i) = db.emb[i]
Base.length(db::MetaVectorDatabase) = length(db.emb)
fetchfield(db::MetaVectorDatabase, field) = [get(r, field, missing) for r in db.meta]
function fetchfield(db::MetaVectorDatabase, id, field)
    get(db.meta[id], field, missing)
end


fetchfield (generic function with 2 methods)

In [82]:
corpus = let
    filename = repolist[1]
    MetaVectorDatabase(filename)
end

MetaVectorDatabase{Vector{JSON3.Object{Base.CodeUnits{UInt8, String}, Vector{UInt64}}}, MatrixDatabase{Matrix{Float32}}}(JSON3.Object{Base.CodeUnits{UInt8, String}, Vector{UInt64}}[{
     "publisher": [],
      "language": [
                    "spa"
                  ],
       "subject": [
                    "",
                    "info:eu-repo/classification/lcc/Matemáticas--Estudio y enseñanza--Investigación",
                    "info:eu-repo/classification/lcc/Matemáticas--Aspectos sociales",
                    "info:eu-repo/classification/lcc/Observación en el aula",
                    "info:eu-repo/classification/cti/1"
                  ],
          "repo": "buap",
   "description": [
                    "\"Es entendible que, después de muchos años de aprendizaje memorístico e imitativo y correspondientes rutinas formadas, en un año no se pudieran generar cambios significativos en las creencias de los estudiantes. Sin embargo, hay claros cambios positivos en algunos de ello

In [83]:
length(corpus)

10141

In [85]:
#record(corpus, "meta/oai:repositorioinstitucional.buap.mx:20.500.12371/100")

In [86]:
#[norm(col) for col in corpus]

IDX = let dist = NormalizedCosineDistance()
    ExhaustiveSearch(; dist, db=corpus)
end

ExhaustiveSearch{NormalizedCosineDistance, MetaVectorDatabase{Vector{JSON3.Object{Base.CodeUnits{UInt8, String}, Vector{UInt64}}}, MatrixDatabase{Matrix{Float32}}}}(NormalizedCosineDistance(), MetaVectorDatabase{Vector{JSON3.Object{Base.CodeUnits{UInt8, String}, Vector{UInt64}}}, MatrixDatabase{Matrix{Float32}}}(JSON3.Object{Base.CodeUnits{UInt8, String}, Vector{UInt64}}[{
     "publisher": [],
      "language": [
                    "spa"
                  ],
       "subject": [
                    "",
                    "info:eu-repo/classification/lcc/Matemáticas--Estudio y enseñanza--Investigación",
                    "info:eu-repo/classification/lcc/Matemáticas--Aspectos sociales",
                    "info:eu-repo/classification/lcc/Observación en el aula",
                    "info:eu-repo/classification/cti/1"
                  ],
          "repo": "buap",
   "description": [
                    "\"Es entendible que, después de muchos años de aprendizaje memorístico e imitati

In [87]:

function render_multi(preprocessing::Function, f)
    if length(f) == 1
        h.span(preprocessing(f[1]))
    else
        h.ul([h.li(preprocessing(t)) for t in f]...)
    end
end

render_multi(f) = render_multi(identity, f)

function preprocessing_desc(t::AbstractString)
    t = replace(t, r"^\W+"imx => "")
    t1 = replace(t, r"[\W\s\.]+$"imx => "")
    # startswith(t, "Bene") || @assert t != t1 repr(t)
    t1
end

function render_record!(L, r)
    title = h.tr(style="background-color: rgb(80, 140, 140);",
        h.td("título", class="left-cell"), h.td(r["title"], class="right-cell"))
    author = h.tr(h.td("autor(es)", class="left-cell"), h.td(render_multi(titlecase, r["creator"]), class="right-cell"))
    desc = h.tr(h.td("desc", class="left-cell"), h.td(render_multi(preprocessing_desc, r["description"]), class="right-cell"))
    repo = h.tr(h.td("repo", class="left-cell"), h.td(r["repo"], class="right-cell"))
    key = h.tr(h.td("key", class="left-cell"), h.td(r["key"], class="right-cell"))
    date = h.tr(h.td("fecha", class="left-cell"), h.td(r["date"], class="right-cell"))
    pub = h.tr(h.td("editorial", class="left-cell"), h.td(r["publisher"], class="right-cell"))
    
    push!(L, title)
    push!(L, author)
    push!(L, desc)
    push!(L, repo)
    push!(L, pub)
    push!(L, key)
end

function render_result(corpus, qtext, res)
    L = []
    for (i, p) in enumerate(viewitems(res))
        r = record(corpus, p.id)
        render_record!(L, r)
    end

    #(string(h.table(h.tr(h.th("nombre"), h.th("valor")), L...)))
    h.div(
        h.h2("resultados para '$qtext'"),
        h.table(L...)
    )
end

render_result (generic function with 1 method)

In [88]:
ctx = GenericContext()
res = knnqueue(ctx, 10)
qID = 11
qtext = only(fetchfield(corpus, qID, "title"))
search(IDX, ctx, corpus[qID], res)

KnnSorted{Vector{IdWeight}}(IdWeight[IdWeight(0x0000000b, 3.5762787f-7), IdWeight(0x00000434, 0.4293083f0), IdWeight(0x00002528, 0.45150948f0), IdWeight(0x00002722, 0.45696795f0), IdWeight(0x00000f15, 0.4662496f0), IdWeight(0x00000a46, 0.47492743f0), IdWeight(0x000015ae, 0.48515558f0), IdWeight(0x000008a8, 0.4859377f0), IdWeight(0x000024e3, 0.4889804f0), IdWeight(0x00000f22, 0.491706f0)], 1, 10, 10, 10141, 0)

In [89]:
render_result(corpus, qtext, res)

título,Atribución de autoría combinando información léxico - sintáctica mediante representaciones holográficas reducidas
autor(es),Jovany Marcos Ramirez
desc,"El presente trabajo, busca determinar si la tarea de atribucion de autorıa puede beneficiarse, con la combinacion de caracterısticas extraıdas de textos, de diferente nivel gramatical. De acuerdo a los trabajos revisados, las caracterısticas lexicas y en particular el uso de n-gramas de caracteres han producido resultados para la atribucion de autorıa con un 76 % de precision en promedio. Se ha visto que los trabajos en los que se utilizan caracterısticas sintacticas como son n-gramas sintacticos (sn-gramas) han obtenido un 95 % de precision en corpus de 39 documentos; las gramaticas libres de contexto probabilısticas han producido resultados con precision de 83 % en corpus de 120 documentos. Estos resultados nos indican que la utilizacion de aspectos sintacticos, utilizando representaciones textuales novedosas puede conducir a la obtencion de resultados aceptables. En particular, en este trabajo se utiliza la representacion holografica reducida para combinar informacion lexica y sintactica, representar textos de diferentes autores y comprobar si dicha representacion contribuye a mejorar la precision de la tarea de atribucion de autorıa"
repo,buap
editorial,
key,meta/oai:repositorioinstitucional.buap.mx:20.500.12371/10016
título,Identificación de perfiles de usuario
autor(es),Patricia Maria Espinoza Fong
desc,"La gran cantidad de conversaciones que ocurren día a día en la web ha propiciado una nueva forma de búsqueda de víctimas por parte de persona maliciosas que intentan, interactuar fundamentalmente con niños o jóvenes. En la web se puede obtener todo tipo de información acerca de una persona sin necesidad de tener contacto físico con ella. Una persona puede crear un perfil falso y establecer todo tipo de conversación con individuos de menor edad sin que esto pueda detectarse. En este trabajo de tesis se experimenta con distintos métodos para la correcta representación de un autor dado en el problema de atribución de autoría. El objetivo es determinar un conjunto de características (novedosas) tanto léxicas como sintácticas y semánticas capaces de representar el estilo de escritura de un autor con respecto a otros de la manera más fiel posible"
repo,buap
editorial,


In [92]:

#=
subject => ["info:eu-repo/classification/Autor/Multilingual sentiment analysis", "info:eu-repo/classification/Autor/Error-robust text representations", "info:eu-repo/classification/Autor/Opinion mining", "info:eu-repo/classification/cti/7", "info:eu-repo/classification/cti/33", "info:eu-repo/classification/cti/3304", "info:eu-repo/classification/cti/120304", "info:eu-repo/classification/cti/120304"]
repo => centrogo
description => ["Recently, sentient analysis has received a lot of attention due to the interest in mining opinions of social media users. Sentiment analysis consists in determining the polarity of a given text, i.e., its degree of positiveness or negativeness. Traditionally, Sentiment Analysis algorithms have been tailored to a specific language given the complexity of having a number of lexical variations and errors introduced by the people generating content. In this contribution, our aim is to provide a simple to implement and easy to use multilingual framework, that can serve as a baseline for sentiment analysis contests, and as a starting point to build new sentiment analysis systems. We compare our approach in eight different languages, three of them correspond to important international contests, namely, SemEval (English), TASS (Spanish), and SENTIPOLC (Italian). Within the competitions, our approach reaches from medium to high positions in the rankings; whereas in the remaining languages our approach outperforms the reported results."]
key => meta/oai:centrogeo.repositorioinstitucional.mx:1012/244
creator => ["Eric Tellez", "SABINO MIRANDA JIMENEZ", "Mario Graff", "Daniela Moctezuma", "Ranyart Rodrigo Suarez Ponce de Leon", "Oscar Sánchez Siordia"]
date => ["2017-07-15", "info:eu-repo/date/embargoEnd/2019-07-16"]
contributor => Union{}[]
rights => ["info:eu-repo/semantics/embargoedAccess", "http://creativecommons.org/licenses/by-nc-nd/4.0"]
file => nothing
title => ["A simpl
e approach to multilingual polari

ty classification in twitter"]
identifier => ["oai:centrogeo.repositorioinstitucional.mx:1012/244", "http://centrogeo.repositorioinstitucional.mx/jspui/handle/1012/244"]
type => 
=#

In [93]:
#tokenize(qtext)

In [94]:
qtext = "análisis de sentimiento y mineria de opinion" 
model, q = embed(Float32, qtext)
normalize!(q)
reuse!(res)
search(IDX, ctx, q, res)
render_result(corpus, qtext, res)

título,Combinación de clasificadores para el análisis de sentimientos
autor(es),
desc,"Hoy en día es muy común encontrar en redes sociales, blogs, microblogs, páginas web, entre otras, información u opiniones de los usuarios que expresan su punto de vista en internet acerca de algo. Dicho fenómeno ha generado interés por el análisis de sentimientos (AS), un área del procesamiento de lenguaje natural (NLP) que se encarga de identificar opiniones relacionadas con un objeto. El interés proviene, de que un factor determinante para la toma de decisión de las personas es precisamente la opinión, por ejemplo cuando compramos algún producto en internet, queremos conocer la opinión de los demás acerca del producto que deseamos adquirir. De la misma manera, las empresas tienen como objetivo encontrar indicadores que contribuyan a cubrir las necesidades de sus clientes, mejorando sus productos y servicios, lanzando al mercado productos que con base en las opiniones adquiridas prefieran o soliciten sus clientes y estar al pendiente de su posición con respecto a la competencia. En el ambiente político es importante Conocer la opinión de las personalidades públicas, elegir la propaganda idónea según las preferencias u opiniones de la gente o simplemente elegir el producto mejor valorado por los usuariosBenemérita Universidad Autónoma de Puebla"
repo,buap
editorial,
key,meta/oai:repositorioinstitucional.buap.mx:20.500.12371/8501
título,Percepción y emociones en la ciudad de Puebla ante las coaliciones electorales. Nuevas tendencias en neuromarketing
autor(es),
desc,"En México, los estudios de opinión pública representan una excelente oportunidad para conocer la percepción ciudadana respecto a los asuntos públicos (Pérez-Verduzco, 2019). Es por ello que, esa percepción puede ser una gran influencia a la hora de tomar acciones con respecto al voto, más aún si durante muchos años los partidos políticos han sido antagónicos y finalmente deciden unirse en coalición para enfrentar a la oposición. Así mismo, el conocer la percepción ciudadana respecto a las coaliciones es una manera de poder entender las emociones que puede generar en ellos los cambios o acuerdos que estas representan entre partidos. El analizar qué los ha movido en las elecciones del 2018 a depositar su voto en las urnas es lo que finalmente dará pie al análisis comparativo, para conocer qué fue lo que motivó a los ciudadanos, así como las emociones que causaba en ellos tanto el candidato, el partido o la coalición a la que pertenecíaBenemérita Universidad Autónoma de Puebla"
repo,buap
editorial,
